# ClearSight V4: Professional Re-Identification (ReID) Pipeline

This pipeline uses **YOLOv8 + ByteTrack** to track human bodies, and **FaceNet** to bind the target's identity to a specific body tracker ID. This guarantees 100% lock-on even if the target turns around or their face becomes blurry.

In [5]:
import cv2
import torch
import numpy as np
from torchvision import transforms
import torch.nn.functional as F
import facenet_pytorch
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt
from IPython.display import Video

### 1. Initialize PyTorch AI Engines

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# 1. Identity Verification Model (FaceNet + MTCNN)
face_detector = facenet_pytorch.MTCNN(thresholds=[0.6, 0.7, 0.7], keep_all=True, device=device, min_face_size=10)
resnet = facenet_pytorch.InceptionResnetV1(pretrained='vggface2').eval().to(device)
preprocess = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((160, 160)),
    transforms.ToTensor(), transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 2. Body Tracking Model (YOLOv8)
yolo_model = YOLO('yolov8n.pt')

Running on: cuda


### 2. Generate Master Vector (Target Identity)

In [7]:
def get_raw_face(img_rgb, box):
    x1, y1, x2, y2 = box.astype(int)
    w, h = x2 - x1, y2 - y1
    x, y = max(0, x1), max(0, y1)
    face_crop = img_rgb[y:y+h, x:x+w]
    if face_crop.size == 0: return None
    return face_crop

target_image_path = 'C:/Users/rajti/Downloads/messi1.jpg'
img = Image.open(target_image_path).convert('RGB')
img_rgb = np.array(img)

boxes, probs = face_detector.detect(img_rgb)
if boxes is not None and len(boxes) > 0:
    biggest_box = max(boxes, key=lambda b: (b[2]-b[0]) * (b[3]-b[1]))
    perfect_face = get_raw_face(img_rgb, biggest_box)
    tensor = preprocess(perfect_face).unsqueeze(0).to(device)
    with torch.no_grad():
        master_embedding = resnet(tensor)
    print("Master Vector ready.")
else:
    print("FAILED to find face in target image.")

Master Vector ready.


### 3. Run ReID Tracking Pipeline on Target Video

In [8]:
video_path = 'C:/Users/rajti/Downloads/messivideo.mp4'
out_path = 'messivideo_ReID_tracked.mp4'

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

TARGET_TRACK_ID = None
known_non_targets = set()
COSINE_THRESHOLD = 0.15

results = yolo_model.track(source=video_path, classes=[0], stream=True, persist=True, verbose=False)

print("Tracking Video...")
for r in results:
    frame_bgr = r.orig_img
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    
    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy().astype(int)
        
        for box, track_id in zip(boxes, track_ids):
            if track_id == TARGET_TRACK_ID:
                x1, y1, x2, y2 = box.astype(int)
                cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 3)
                cv2.putText(frame_bgr, f"LOCKED RE-ID [{track_id}]", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                continue
                
            if track_id in known_non_targets:
                continue
                
            if TARGET_TRACK_ID is None:
                x1, y1, x2, y2 = box.astype(int)
                x, y = max(0, x1), max(0, y1)
                w, h = x2 - x1, y2 - y1
                y_exp = max(0, int(y - h * 0.2))
                body_crop = frame_rgb[y_exp:y+h, x:x+w]
                if body_crop.size == 0: continue
                
                face_boxes, probs = face_detector.detect(body_crop)
                if face_boxes is not None and len(face_boxes) > 0:
                    biggest_face = max(face_boxes, key=lambda b: (b[2]-b[0]) * (b[3]-b[1]))
                    perfect_face = get_raw_face(body_crop, biggest_face)
                    if perfect_face is not None:
                        with torch.no_grad():
                            tensor = preprocess(perfect_face).unsqueeze(0).to(device)
                            embedding = resnet(tensor)
                        
                        cosine_sim = F.cosine_similarity(master_embedding, embedding).item()
                        if cosine_sim > COSINE_THRESHOLD:
                            TARGET_TRACK_ID = track_id
                            print(f"Target Identified! Bound to Body ID #{track_id} (Confidence: {cosine_sim:.2f})")
                            cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 3)
                            cv2.putText(frame_bgr, f"TARGET ACQUIRED [{track_id}]", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        else:
                            known_non_targets.add(track_id)
    
    out.write(frame_bgr)

cap.release()
out.release()
print(f"Done! Video saved to {out_path}")

Tracking Video...
Target Identified! Bound to Body ID #63 (Confidence: 0.21)
Done! Video saved to messivideo_ReID_tracked.mp4
